# Análisis de resultados — API de Gemini (texto + visión)

Notebook con el **análisis de resultados** de las dos prácticas del repositorio:

1. **Práctica 1 (`gemini-text.py`)**: 10 textos analizados con 4 tareas de NLP (detección de idioma, entidades nombradas, sentimiento y resumen) mediante prompts estructurados con respuesta en JSON.
2. **Práctica 2 (`gemini-images.py`)**: 4 imágenes + 1 PDF analizados con 4 tareas visuales (OCR, descripción de contenido visual, clasificación de escenas y detección de objetos) con prompts multimodales y respuesta en JSON.

Este notebook **no llama a la API**: solo lee los resultados ya generados (`results/informe.json`, `results/informe_imagenes.json`, `results/imagenes-analisis-1.json` y `results/analisis-1.json`) y los presenta en tablas, rankings y conclusiones.

> Para regenerar los datos: `python gemini-text.py` y `python gemini-images.py`.

In [ ]:
import json
from pathlib import Path

import pandas as pd

BASE = Path.cwd()
if not (BASE / "results").exists():
    BASE = Path("..")  # si el notebook se abre desde una subcarpeta

def cargar(nombre: str) -> dict:
    ruta = BASE / "results" / nombre
    if not ruta.exists():
        raise FileNotFoundError(f"Falta {ruta}: ejecuta primero el script correspondiente")
    return json.loads(ruta.read_text(encoding="utf-8"))

texto = cargar("informe.json")
vision = cargar("informe_imagenes.json")
mt, mv = texto["metricas"], vision["metricas"]

print(f"Práctica 1 (texto):  {mt['textos_evaluados']}/{mt['textos_totales']} textos  | modelo {texto['modelo']}")
print(f"Práctica 2 (visión): {mv['imagenes_evaluadas']}/{mv['imagenes_totales']} archivos | modelo {vision['modelo']}")

## 1. Práctica de texto: métricas globales por tarea

In [ ]:
resumen_texto = pd.DataFrame([
    {"Tarea": "Detección de idioma", "Métrica": "Exactitud", "Valor": mt["idioma"]["precision"]},
    {"Tarea": "Análisis de sentimiento", "Métrica": "Exactitud", "Valor": mt["sentimiento"]["precision"]},
    {"Tarea": "Extracción de entidades", "Métrica": "Precisión / Recall / F1", "Valor": mt["entidades"]["f1"]},
    {"Tarea": "Resumen automático", "Métrica": "ROUGE-1 F1", "Valor": mt["resumen"]["rouge1_f1"]},
])
resumen_texto["Precisión (%)"] = (resumen_texto.pop("Valor") * 100).round(1)
display(resumen_texto.sort_values("Precisión (%)", ascending=False).reset_index(drop=True))

In [ ]:
df_tipo_texto = pd.DataFrame.from_dict(mt["por_tipo"], orient="index")
df_tipo_texto.index.name = "tipo de texto"
porcentaje = [c for c in df_tipo_texto.columns if c != "textos"]
display((df_tipo_texto[porcentaje] * 100).round(1).join(df_tipo_texto["textos"]))

In [ ]:
detalle_texto = pd.DataFrame([
    {
        "texto": d["fichero"],
        "tipo": d["tipo"],
        "idioma": "OK" if d["idioma"]["correcto"] else f"{d['idioma']['predicho']} ≠ {d['idioma']['esperado']}",
        "sentimiento": "OK" if d["sentimiento"]["correcto"] else f"{d['sentimiento']['predicho']} ≠ {d['sentimiento']['esperado']}",
        "entidades F1": round(d["entidades"]["f1"] * 100, 1),
        "resumen ROUGE-1": round(d["resumen"]["rouge1_f1"] * 100, 1),
    }
    for d in texto["detalles"]
])
display(detalle_texto)

## 2. Práctica de visión (imágenes + PDF): métricas globales por tarea visual

In [ ]:
resumen_vision = pd.DataFrame([
    {"Tarea visual": "OCR (extracción de texto)", "Métrica": "F1 de tokens", "Valor": mv["ocr"]["f1"]},
    {"Tarea visual": "Descripción de contenido", "Métrica": "ROUGE-1 F1", "Valor": mv["descripcion"]["rouge1_f1"]},
    {"Tarea visual": "Clasificación de escenas", "Métrica": "Exactitud", "Valor": mv["escena"]["precision"]},
    {"Tarea visual": "Detección de objetos", "Métrica": "Precisión / Recall / F1", "Valor": mv["objetos"]["f1"]},
])
resumen_vision["Precisión (%)"] = (resumen_vision.pop("Valor") * 100).round(1)
display(resumen_vision.sort_values("Precisión (%)", ascending=False).reset_index(drop=True))

objetos = mv["objetos"]
print(
    f"Objetos: precisión {objetos['precision'] * 100:.1f}% | "
    f"recall {objetos['recall'] * 100:.1f}% | F1 {objetos['f1'] * 100:.1f}%"
)

In [ ]:
df_tipo_vision = pd.DataFrame.from_dict(mv["por_tipo"], orient="index")
df_tipo_vision.index.name = "tipo de archivo"
cols_pct = [c for c in df_tipo_vision.columns if c != "imagenes"]
display((df_tipo_vision[cols_pct] * 100).round(1).join(df_tipo_vision["imagenes"]))

In [ ]:
detalle_vision = pd.DataFrame([
    {
        "archivo": d["fichero"],
        "tipo": d["tipo"],
        "OCR F1": round(d["ocr"]["f1"] * 100, 1),
        "escena": f"{d['escena']['predicha']} / {d['escena']['esperada']} "
                  + ("OK" if d["escena"]["correcta"] else "FALLO"),
        "objetos F1": round(d["objetos"]["f1"] * 100, 1),
        "descripción ROUGE-1": round(d["descripcion"]["rouge1_f1"] * 100, 1),
    }
    for d in vision["detalles"]
])
display(detalle_vision)

## 3. Ranking comparativo de las 8 tareas (texto + visión)

In [ ]:
ranking = pd.DataFrame([
    {"Práctica": "texto", "Tarea": "Detección de idioma", "Valor (%)": mt["idioma"]["precision"] * 100},
    {"Práctica": "texto", "Tarea": "Sentimiento", "Valor (%)": mt["sentimiento"]["precision"] * 100},
    {"Práctica": "texto", "Tarea": "Entidades (F1)", "Valor (%)": mt["entidades"]["f1"] * 100},
    {"Práctica": "texto", "Tarea": "Resumen (ROUGE-1)", "Valor (%)": mt["resumen"]["rouge1_f1"] * 100},
    {"Práctica": "visión", "Tarea": "OCR (F1)", "Valor (%)": mv["ocr"]["f1"] * 100},
    {"Práctica": "visión", "Tarea": "Escenas", "Valor (%)": mv["escena"]["precision"] * 100},
    {"Práctica": "visión", "Tarea": "Objetos (F1)", "Valor (%)": mv["objetos"]["f1"] * 100},
    {"Práctica": "visión", "Tarea": "Descripción (ROUGE-1)", "Valor (%)": mv["descripcion"]["rouge1_f1"] * 100},
])
ranking["Valor (%)"] = ranking["Valor (%)"].round(1)
display(ranking.sort_values("Valor (%)", ascending=False).reset_index(drop=True))

try:
    import matplotlib.pyplot as plt

    ordenado = ranking.sort_values("Valor (%)")
    colores = {"texto": "#4285F4", "visión": "#34A853"}
    ax = ordenado.plot.barh(
        x="Tarea", y="Valor (%)", legend=False,
        color=ordenado["Práctica"].map(colores), figsize=(9, 5),
    )
    ax.set_xlim(0, 100)
    ax.set_title("Precisión por tarea (API de Gemini)")
    for i, v in enumerate(ordenado["Valor (%)"]):
        ax.text(v + 1, i, f"{v:.1f}%", va="center")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib no está instalado (opcional): pip install matplotlib")

## 4. Ejemplo de respuesta estructurada (JSON) del modelo

In [ ]:
datos_vision = cargar("imagenes-analisis-1.json")
ejemplo = next(
    (r for r in datos_vision["resultados"] if r.get("ocr", {}).get("texto")),
    datos_vision["resultados"][0],
)
print(f"Archivo: {ejemplo['fichero']}")
print(json.dumps({k: v for k, v in ejemplo.items() if k != "fichero"}, ensure_ascii=False, indent=2))

## 5. Conclusiones

In [ ]:
mejor_texto = resumen_texto.loc[resumen_texto["Precisión (%)"].idxmax()]
peor_texto = resumen_texto.loc[resumen_texto["Precisión (%)"].idxmin()]
mejor_vision = resumen_vision.loc[resumen_vision["Precisión (%)"].idxmax()]
peor_vision = resumen_vision.loc[resumen_vision["Precisión (%)"].idxmin()]

conclusiones = [
    f"**Texto**: la tarea más precisa es {mejor_texto['Tarea']} ({mejor_texto['Precisión (%)']}%) "
    f"y la menos precisa {peor_texto['Tarea']} ({peor_texto['Precisión (%)']}%).",
    f"**Visión**: la tarea más precisa es {mejor_vision['Tarea visual']} ({mejor_vision['Precisión (%)']}%) "
    f"y la menos precisa {peor_vision['Tarea visual']} ({peor_vision['Precisión (%)']}%).",
    f"Las tareas de **categoría** (idioma, sentimiento, escenas) alcanzan la mayor exactitud; "
    f"las de **generación/expresión libre** (resumen y descripción, medidas con ROUGE-1) son las más exigentes.",
    f"En OCR solo hay dificultad real con texto pequeño o en idioma no latino (el PDF de 46 páginas "
    f"se transcribe con F1 {mv['ocr']['f1'] * 100:.1f}% sobre su primera página).",
    f"El JSON estructurado (`response_format` + esquema) evita errores de formato en las "
    f"{mt['textos_evaluados'] * 4 + mv['imagenes_evaluadas'] * 4} llamadas realizadas.",
]

from IPython.display import Markdown, display
display(Markdown("\n\n".join(f"- {c}" for c in conclusiones)))